In [1]:
"""
WESAD Binary Stress Detection - Notebook 3: LOSO Training (FULL COMPLIANCE)
============================================================================
Complete implementation with:
- Detailed fold-by-fold logging
- Model architecture printing
- Training progress per epoch
- Per-fold metrics and plots
- Training curve visualization
"""

import numpy as np
import pickle
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks, regularizers
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

In [2]:
# CONFIGURATION

MODEL_CONFIG = {
    'chest_modalities': ['ECG', 'EDA', 'EMG', 'Respiration', 'Temperature'],
    'wrist_modalities': ['ACC', 'BVP', 'EDA', 'Temperature'],
    'hrv_features': ['mean_rr', 'sdnn', 'rmssd', 'pnn50', 'lf_power', 'hf_power', 'lf_hf_ratio'],
    'cnn_filters_1': 64, 'cnn_kernel_1': 5,
    'cnn_filters_2': 128, 'cnn_kernel_2': 3,
    'pool_size': 2, 'lstm_units': 64,
    'dense_1': 128, 'dense_2': 64,
    'dropout': 0.5, 'lstm_dropout': 0.3,
    'lstm_recurrent_dropout': 0.2, 'l2_reg': 0.001,
}

TRAINING_CONFIG = {
    'loss': 'binary_crossentropy', 'optimizer': 'adam',
    'learning_rate': 1e-3, 'batch_size': 64,
    'epochs': 30, 'early_stopping_patience': 5,
    'verbose_training': 0,  # Show epoch progress
}

# LOAD DATA

def load_data():
    with open('processed_data/preprocessed_loso_data.pkl', 'rb') as f:
        data = pickle.load(f)
    print(f"✓ Loaded {len(data['data'])} subjects\n")
    return data['data'], data['config']

In [3]:
# DATA PREPARATION

def prepare_chest_data(preprocessed_data, subjects):
    all_signals = {mod: [] for mod in MODEL_CONFIG['chest_modalities']}
    all_labels = []
    
    for subject in subjects:
        if subject not in preprocessed_data:
            continue
        data = preprocessed_data[subject]
        for mod in MODEL_CONFIG['chest_modalities']:
            all_signals[mod].append(data['chest_signals'][mod])
        all_labels.append(data['chest_labels'])
    
    for mod in all_signals:
        all_signals[mod] = np.concatenate(all_signals[mod], axis=0) if all_signals[mod] else np.array([])
    all_labels = np.concatenate(all_labels, axis=0) if all_labels else np.array([])
    
    return all_signals, all_labels

def prepare_wrist_data(preprocessed_data, subjects):
    all_signals = {mod: [] for mod in MODEL_CONFIG['wrist_modalities']}
    all_labels = []
    
    for subject in subjects:
        if subject not in preprocessed_data:
            continue
        data = preprocessed_data[subject]
        for mod in MODEL_CONFIG['wrist_modalities']:
            all_signals[mod].append(data['wrist_signals'][mod])
        all_labels.append(data['wrist_labels'])
    
    for mod in all_signals:
        all_signals[mod] = np.concatenate(all_signals[mod], axis=0) if all_signals[mod] else np.array([])
    all_labels = np.concatenate(all_labels, axis=0) if all_labels else np.array([])
    
    return all_signals, all_labels

def prepare_hrv_data(preprocessed_data, subjects):
    all_features, all_labels = [], []
    
    for subject in subjects:
        if subject not in preprocessed_data:
            continue
        data = preprocessed_data[subject]
        features = []
        for window in data['hrv_features']:
            features.append([window[k] for k in MODEL_CONFIG['hrv_features']])
        all_features.append(np.array(features))
        all_labels.append(data['hrv_labels'])
    
    all_features = np.concatenate(all_features, axis=0) if all_features else np.array([])
    all_labels = np.concatenate(all_labels, axis=0) if all_labels else np.array([])
    
    valid = ~np.isnan(all_features).any(axis=1)
    return all_features[valid], all_labels[valid]

In [4]:
# MODEL ARCHITECTURES

def create_modality_branch(name, input_shape):
    inp = layers.Input(shape=input_shape, name=f'{name}_input')
    x = layers.Conv1D(MODEL_CONFIG['cnn_filters_1'], MODEL_CONFIG['cnn_kernel_1'], padding='same')(inp)
    x = layers.ReLU()(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(MODEL_CONFIG['pool_size'])(x)
    x = layers.Conv1D(MODEL_CONFIG['cnn_filters_2'], MODEL_CONFIG['cnn_kernel_2'], padding='same')(x)
    x = layers.ReLU()(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(MODEL_CONFIG['pool_size'])(x)
    x = layers.LSTM(MODEL_CONFIG['lstm_units'], return_sequences=False,
                    dropout=MODEL_CONFIG['lstm_dropout'],
                    recurrent_dropout=MODEL_CONFIG['lstm_recurrent_dropout'])(x)
    return models.Model(inputs=inp, outputs=x, name=f'{name}_branch')

def create_chest_model():
    branches, inputs, outputs = {}, {}, []
    
    for mod in MODEL_CONFIG['chest_modalities']:
        branch = create_modality_branch(mod, (240, 1))
        branches[mod] = branch
        inputs[mod] = branch.input
        outputs.append(branch.output)
    
    concat = layers.Concatenate()(outputs)
    x = layers.Dense(MODEL_CONFIG['dense_1'], activation='relu',
                     kernel_regularizer=regularizers.l2(MODEL_CONFIG['l2_reg']))(concat)
    x = layers.Dropout(MODEL_CONFIG['dropout'])(x)
    x = layers.Dense(MODEL_CONFIG['dense_2'], activation='relu',
                     kernel_regularizer=regularizers.l2(MODEL_CONFIG['l2_reg']))(x)
    x = layers.Dropout(MODEL_CONFIG['dropout'])(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    
    model = models.Model(inputs=[inputs[m] for m in MODEL_CONFIG['chest_modalities']], outputs=out, name='Chest_Model')
    model.compile(optimizer=optimizers.Adam(TRAINING_CONFIG['learning_rate']),
                  loss=TRAINING_CONFIG['loss'], metrics=['accuracy'])
    return model

def create_wrist_model():
    branches, inputs, outputs = {}, {}, []
    
    for mod in MODEL_CONFIG['wrist_modalities']:
        shape = (240, 3) if mod == 'ACC' else (240, 1)
        branch = create_modality_branch(mod, shape)
        branches[mod] = branch
        inputs[mod] = branch.input
        outputs.append(branch.output)
    
    concat = layers.Concatenate()(outputs)
    x = layers.Dense(MODEL_CONFIG['dense_1'], activation='relu',
                     kernel_regularizer=regularizers.l2(MODEL_CONFIG['l2_reg']))(concat)
    x = layers.Dropout(MODEL_CONFIG['dropout'])(x)
    x = layers.Dense(MODEL_CONFIG['dense_2'], activation='relu',
                     kernel_regularizer=regularizers.l2(MODEL_CONFIG['l2_reg']))(x)
    x = layers.Dropout(MODEL_CONFIG['dropout'])(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    
    model = models.Model(inputs=[inputs[m] for m in MODEL_CONFIG['wrist_modalities']], outputs=out, name='Wrist_Model')
    model.compile(optimizer=optimizers.Adam(TRAINING_CONFIG['learning_rate']),
                  loss=TRAINING_CONFIG['loss'], metrics=['accuracy'])
    return model

def print_model_summary(model, name):
    print(f"\n{'='*70}")
    print(f"{name} ARCHITECTURE")
    print(f"{'='*70}")
    model.summary()
    print(f"Total parameters: {model.count_params():,}")
    print(f"{'='*70}\n")

In [5]:
# TRAINING

def prepare_inputs(signals, modalities):
    inputs = []
    for mod in modalities:
        data = signals[mod]
        if mod == 'ACC':
            inputs.append(data)  # Already (N, 240, 3)
        else:
            inputs.append(data.reshape(-1, 240, 1))
    return inputs

def train_model(model, train_data, train_labels, val_data, val_labels, modalities, fold_num, model_name):
    train_inputs = prepare_inputs(train_data, modalities)
    val_inputs = prepare_inputs(val_data, modalities)
    
    print(f"\nTraining {model_name}...")
    print(f"  Train size: {len(train_labels)} (Stress: {np.sum(train_labels==1)}, Non-stress: {np.sum(train_labels==0)})")
    print(f"  Val size: {len(val_labels)} (Stress: {np.sum(val_labels==1)}, Non-stress: {np.sum(val_labels==0)})")
    
    early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=TRAINING_CONFIG['early_stopping_patience'],
                                        restore_best_weights=True, verbose=1)
    
    history = model.fit(train_inputs, train_labels, validation_data=(val_inputs, val_labels),
                       epochs=TRAINING_CONFIG['epochs'], batch_size=TRAINING_CONFIG['batch_size'],
                       callbacks=[early_stop], verbose=TRAINING_CONFIG['verbose_training'])
    
    print(f"✓ Training complete (stopped at epoch {len(history.history['loss'])})")
    return history

def train_hrv_model(train_features, train_labels, val_features, val_labels, fold_num):
    print(f"\nTraining HRV Model...")
    print(f"  Train size: {len(train_labels)} (Stress: {np.sum(train_labels==1)}, Non-stress: {np.sum(train_labels==0)})")
    print(f"  Val size: {len(val_labels)} (Stress: {np.sum(val_labels==1)}, Non-stress: {np.sum(val_labels==0)})")
    
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_features)
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(train_scaled, train_labels)
    
    print(f"✓ Training complete")
    return model, scaler

In [6]:
def run_loso(preprocessed_data):
    print("\n" + "="*70)
    print("LEAVE-ONE-SUBJECT-OUT (LOSO) CROSS-VALIDATION")
    print("="*70)
    
    all_subjects = list(preprocessed_data.keys())
    print(f"\nTotal subjects: {len(all_subjects)}")
    print(f"Subjects: {all_subjects}\n")
    
    loso_results = []
    
    for fold_num, test_subject in enumerate(all_subjects, 1):
        print("\n" + "="*70)
        print(f"FOLD {fold_num}/{len(all_subjects)} - Test Subject: {test_subject}")
        print("="*70)
        
        # Split
        test_subjects = [test_subject]
        remaining = [s for s in all_subjects if s != test_subject]
        val_size = max(1, len(remaining) // 7)
        val_subjects = remaining[:val_size]
        train_subjects = remaining[val_size:]
        
        print(f"\nTrain subjects ({len(train_subjects)}): {train_subjects}")
        print(f"Val subjects ({len(val_subjects)}): {val_subjects}")
        print(f"Test subject: {test_subject}")
        
        # Verify no overlap
        assert test_subject not in train_subjects and test_subject not in val_subjects
        print("✓ No subject overlap confirmed")
        
        # Prepare data
        chest_train, chest_train_labels = prepare_chest_data(preprocessed_data, train_subjects)
        chest_val, chest_val_labels = prepare_chest_data(preprocessed_data, val_subjects)
        chest_test, chest_test_labels = prepare_chest_data(preprocessed_data, test_subjects)
        
        wrist_train, wrist_train_labels = prepare_wrist_data(preprocessed_data, train_subjects)
        wrist_val, wrist_val_labels = prepare_wrist_data(preprocessed_data, val_subjects)
        wrist_test, wrist_test_labels = prepare_wrist_data(preprocessed_data, test_subjects)
        
        hrv_train, hrv_train_labels = prepare_hrv_data(preprocessed_data, train_subjects)
        hrv_val, hrv_val_labels = prepare_hrv_data(preprocessed_data, val_subjects)
        hrv_test, hrv_test_labels = prepare_hrv_data(preprocessed_data, test_subjects)
        
        print("\nData Summary:")
        print(f"  Chest Train/Val/Test: {len(chest_train_labels)} / {len(chest_val_labels)} / {len(chest_test_labels)}")
        print(f"  Wrist Train/Val/Test: {len(wrist_train_labels)} / {len(wrist_val_labels)} / {len(wrist_test_labels)}")
        print(f"  HRV   Train/Val/Test: {len(hrv_train_labels)} / {len(hrv_val_labels)} / {len(hrv_test_labels)}")
        
        print(f"  Chest Train Class Balance: {np.bincount(chest_train_labels)}")
        print(f"  Wrist Train Class Balance: {np.bincount(wrist_train_labels)}")
        print(f"  HRV   Train Class Balance: {np.bincount(hrv_train_labels)}")
        
        # Train models
        if fold_num == 1:
            chest_model = create_chest_model()
            print_model_summary(chest_model, "CHEST MODEL")
        else:
            chest_model = create_chest_model()
        
        chest_history = train_model(
            chest_model, chest_train, chest_train_labels, 
            chest_val, chest_val_labels, MODEL_CONFIG['chest_modalities'],
            fold_num, "Chest Model"
        )
        
        print(f"  Final Chest Val Accuracy: {chest_history.history['val_accuracy'][-1]:.4f}")
        
        if fold_num == 1:
            wrist_model = create_wrist_model()
            print_model_summary(wrist_model, "WRIST MODEL")
        else:
            wrist_model = create_wrist_model()
        
        wrist_history = train_model(
            wrist_model, wrist_train, wrist_train_labels,
            wrist_val, wrist_val_labels, MODEL_CONFIG['wrist_modalities'],
            fold_num, "Wrist Model"
        )
        
        print(f"  Final Wrist Val Accuracy: {wrist_history.history['val_accuracy'][-1]:.4f}")
        
        hrv_model, hrv_scaler = train_hrv_model(
            hrv_train, hrv_train_labels, hrv_val, hrv_val_labels, fold_num
        )
        
        # Predict
        chest_test_inputs = prepare_inputs(chest_test, MODEL_CONFIG['chest_modalities'])
        chest_pred_proba = chest_model.predict(chest_test_inputs, verbose=0).flatten()
        chest_pred = (chest_pred_proba > 0.5).astype(int)
        
        wrist_test_inputs = prepare_inputs(wrist_test, MODEL_CONFIG['wrist_modalities'])
        wrist_pred_proba = wrist_model.predict(wrist_test_inputs, verbose=0).flatten()
        wrist_pred = (wrist_pred_proba > 0.5).astype(int)
        
        hrv_test_scaled = hrv_scaler.transform(hrv_test)
        hrv_pred = hrv_model.predict(hrv_test_scaled)
        
        # Test accuracy reporting
        chest_acc = np.mean(chest_pred == chest_test_labels)
        wrist_acc = np.mean(wrist_pred == wrist_test_labels)
        hrv_acc = np.mean(hrv_pred == hrv_test_labels)
        
        print("\nTest Performance:")
        print(f"  Chest Accuracy: {chest_acc:.4f}")
        print(f"  Wrist Accuracy: {wrist_acc:.4f}")
        print(f"  HRV   Accuracy: {hrv_acc:.4f}")
        
        # Majority voting
        majority_vote = (
            chest_pred + wrist_pred + hrv_pred >= 2
        ).astype(int)
        
        fusion_acc = np.mean(majority_vote == chest_test_labels)
        print(f"  Majority Voting Accuracy: {fusion_acc:.4f}")
        
        # Store
        loso_results.append({
            'fold': fold_num,
            'test_subject': test_subject,
            'chest_pred': chest_pred,
            'wrist_pred': wrist_pred,
            'hrv_pred': hrv_pred,
            'chest_labels': chest_test_labels,
            'wrist_labels': wrist_test_labels,
            'hrv_labels': hrv_test_labels,
            'chest_history': chest_history.history,
            'wrist_history': wrist_history.history,
        })
        
        print(f"\n✓ Fold {fold_num} complete")
    
    return loso_results


In [7]:
# SAVE

def save_results(loso_results):
    Path('loso_results').mkdir(exist_ok=True)
    with open('loso_results/loso_predictions.pkl', 'wb') as f:
        pickle.dump(loso_results, f)
    print(f"\n✓ Results saved to loso_results/loso_predictions.pkl")

# EXECUTE

if __name__ == "__main__":
    print("\n[1/3] Loading data...")
    preprocessed, config = load_data()
    
    print("[2/3] Running LOSO evaluation...")
    loso_results = run_loso(preprocessed)
    
    print("\n[3/3] Saving results...")
    save_results(loso_results)
    
    print("\n" + "="*70)
    print("✓ NOTEBOOK 3 COMPLETE")
    print("="*70)
    print(f"Total folds completed: {len(loso_results)}")
    print("All models trained with detailed logging")
    print("Proceed to Notebook 4 for evaluation and fusion")
    print("="*70)


[1/3] Loading data...
✓ Loaded 15 subjects

[2/3] Running LOSO evaluation...

LEAVE-ONE-SUBJECT-OUT (LOSO) CROSS-VALIDATION

Total subjects: 15
Subjects: ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']


FOLD 1/15 - Test Subject: S2

Train subjects (12): ['S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']
Val subjects (2): ['S3', 'S4']
Test subject: S2
✓ No subject overlap confirmed

Data Summary:
  Chest Train/Val/Test: 817 / 133 / 65
  Wrist Train/Val/Test: 817 / 133 / 65
  HRV   Train/Val/Test: 817 / 133 / 65
  Chest Train Class Balance: [573 244]
  Wrist Train Class Balance: [573 244]
  HRV   Train Class Balance: [573 244]

CHEST MODEL ARCHITECTURE


Model: "Chest_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ ECG_input           │ (None, 240, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ EDA_input           │ (None, 240, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ EMG_input           │ (None, 240, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Respiration_input   │ (None, 240, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Temperature_input   │ (None, 240, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 240, 64)   │        384 │ ECG_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 240, 64)   │        384 │ EDA_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 240, 64)   │        384 │ EMG_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 240, 64)   │        384 │ Respiration_inpu… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 240, 64)   │        384 │ Temperature_inpu… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 240, 64)   │          0 │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_2 (ReLU)      │ (None, 240, 64)   │          0 │ conv1d_2[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_4 (ReLU)      │ (None, 240, 64)   │          0 │ conv1d_4[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_6 (ReLU)      │ (None, 240, 64)   │          0 │ conv1d_6[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_8 (ReLU)      │ (None, 240, 64)   │          0 │ conv1d_8[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 240, 64)   │        256 │ re_lu[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 240, 64)   │        256 │ re_lu_2[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 240, 64)   │        256 │ re_lu_4[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 240, 64)   │        256 │ re_lu_6[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 240, 64)   │        256 │ re_lu_8[0][0]     │
│ (BatchNormalizatio… │                   │            │                 

 Total params: 425,729 (1.62 MB)

 Trainable params: 423,809 (1.62 MB)

 Non-trainable params: 1,920 (7.50 KB)

... (3 outputs left out)

 Total params: 342,913 (1.31 MB)

 Trainable params: 341,377 (1.30 MB)

 Non-trainable params: 1,536 (6.00 KB)

Total parameters: 342,913


Training Wrist Model...
  Train size: 817 (Stress: 244, Non-stress: 573)
  Val size: 133 (Stress: 39, Non-stress: 94)
Epoch 8: early stopping
Restoring model weights from the end of the best epoch: 3.
✓ Training complete (stopped at epoch 8)
  Final Wrist Val Accuracy: 0.7970

Training HRV Model...
  Train size: 817 (Stress: 244, Non-stress: 573)
  Val size: 133 (Stress: 39, Non-stress: 94)
✓ Training complete

Test Performance:
  Chest Accuracy: 0.8769
  Wrist Accuracy: 0.9846
  HRV   Accuracy: 0.7077
  Majority Voting Accuracy: 0.9692

✓ Fold 1 complete

FOLD 2/15 - Test Subject: S3

Train subjects (12): ['S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']
Val subjects (2): ['S2', 'S4']
Test subject: S3
✓ No subject overlap confirmed

Data Summary:
  Chest Train/Val/Test: 817 / 131 / 67
  Wrist Train/Val/Test: 817 / 131 / 67
  HRV   Train/Val/Test: 817 / 131 / 67
  Chest Train Class Balance: [573 244]
  Wrist Train Class Balance: